## Image To Quiz

In [1]:
from glob import glob
import json
from openai import OpenAI
from dotenv import load_dotenv
import os
import base64
from pprint import pprint

load_dotenv()
client = client = OpenAI()

In [2]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

### 프롬프트 수정
예제에 나온대로 하니, 정답을 다 4번으로 만들고, 생뚱맞은 보기를 만들어서 다음과 같은 프로프트를 수정해봤다.

In [3]:
def image_quiz(image_path, n_trial=0, max_trial=3):
    if n_trial >= max_trial: # 최대 시도 회수에 도달하면 포기
        raise Exception("Failed to generate a quiz.")
    
    base64_image = encode_image(image_path) # 이미지를 base64로 인코딩

    quiz_prompt = """
    제공된 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어주세요. 
    정답은 1~4 중 하나만 해당하도록 출제하세요.
    토익 리스닝 문제 스타일로 문제를 만들어주세요.

    [문제 규칙]
    1. 이미지에 실제로 보이는 사람, 사물, 행동, 위치, 색상, 의복 등의 정보를 사용하세요.
    2. 이미지에서 확인할 수 없는 내용을 임의로 만들어내지 마세요.
    3. 선택지는 모두 이미지와 관련된 자연스러운 내용이어야 합니다.
    4. 4개의 선택지 중 단 하나만 틀린 설명이 되도록 만드세요.
    5. 정답 번호는 반드시 (1)~(4) 중 하나를 선택하세요.
    6. 정답 번호는 매번 동일하게 하지 말고, (1)~(4)가 골고루 나오도록 하세요.
    7. 특히 (4)를 정답으로 고정하지 마세요.
    8. 오답은 이미지와 전혀 관계없는 생뚱맞은 내용을 만들지 마세요.
    대신 이미지에 실제로 존재하는 사람, 사물, 행동 등을 이용하여
    색상, 위치, 행동, 개수 등의 세부사항을 살짝 틀리게 만드세요.
    9. 오답은 얼핏 보면 맞는 것처럼 자연스럽게 만들어야 합니다.
    10. 정답을 제외한 나머지 3개의 선택지는 이미지와 정확히 일치해야 합니다.
    11. 정답 번호를 결정한 후, 반드시 해당 선택지만 이미지와 실제로 일치하지 않는지 다시 확인하세요.

    [문제 형식]
    아래는 예시입니다. 
    ----- 예시 -----

    Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 베이커리에서 사람들이 빵을 사고 있는 모습이 담겨 있습니다.
    - (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입고 있습니다.
    - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
    - (4) 점원은 노란색 티셔츠를 입고 있습니다.

    Listening: Which of the following descriptions of the image is incorrect?
    - (1) It shows people buying bread at a bakery.
    - (2) The person standing at the front is wearing a red shirt.
    - (3) There are people lining up to take a train.
    - (4) The clerk is wearing a yellow T-shirt.
        
    정답: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입고 있습니다.
    (주의: 정답은 1~4 중 하나만 선택되도록 출제하세요.)

    [오답 생성 방법]
    오답은 다음과 같은 방식으로 만들어주세요.

    - 실제 사람의 옷 색상을 다른 색상으로 바꾸기
    - 실제 사물의 위치를 바꾸기
    - 실제 사람이 하는 행동을 다른 행동으로 바꾸기
    - 실제 사물의 개수나 상태를 살짝 다르게 표현하기
    - 실제 존재하는 사물이나 사람을 다른 것으로 표현하기

    단, 이미지에 존재하지 않는 전혀 엉뚱한 사물이나 상황을 오답으로 만들지 마세요.

    [중요]
    문제를 만들기 전에 먼저 이미지에서 확인 가능한 주요 요소를 파악하세요.
    그 후 그 요소를 바탕으로 3개의 정확한 설명과 1개의 자연스러운 오답을 만드세요.

    정답은 반드시 (1), (2), (3), (4) 중 하나여야 하며,
    정답 위치는 특정 번호에 편향되지 않도록 하세요.
    ======
    """

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": quiz_prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                    },
                },
            ],
        }
    ]

    try: 
        response = client.chat.completions.create(
            model="gpt-5.6-luna",  # 응답 생성에 사용할 모델 지정
            messages=messages # 대화 기록을 입력으로 전달
        )
    except Exception as e:
        print("failed\n" + e)
        return image_quiz(image_path, n_trial+1)
    
    content = response.choices[0].message.content

    if "Listening:" in content:
        return content, True
    else:
        return image_quiz(image_path, n_trial+1)

In [4]:
q = image_quiz("img/busan_dive.jpg")
pprint(q)

('Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?\n'
 '\n'
 '- (1) 넓은 행사장 안에 많은 사람들이 테이블에 앉아 있습니다.\n'
 '- (2) 행사장에 있는 대부분의 참가자들은 서서 노트북을 사용하고 있습니다.\n'
 '- (3) 여러 테이블 위에 노트북과 개인 소지품이 놓여 있습니다.\n'
 '- (4) 뒤쪽 화면에 “DIVE 2024 IN BUSAN”이라는 문구가 보입니다.\n'
 '\n'
 'Listening: Which of the following descriptions of the image is incorrect?\n'
 '\n'
 '- (1) Many people are seated at tables in a large event hall.\n'
 '- (2) Most of the participants in the hall are standing while using '
 'laptops.\n'
 '- (3) Laptops and personal belongings are placed on several tables.\n'
 '- (4) The words “DIVE 2024 IN BUSAN” can be seen on a screen in the back.\n'
 '\n'
 '정답: **(2)** 대부분의 참가자들은 서 있는 것이 아니라 테이블에 앉아 있습니다.',
 True)


영어 문제 부분만 추출하기 위해 json 파일로 저장

In [5]:
txt = '' # 문제들을 계속 붙여 나가기 위해 빈 문자열 선언
eng_dict = []
no = 1 # 문제 번호를 위해 선언
for g in glob('img/*.jpg'):  # img 파일 하위에 있는 이미지 모두 가져오기
    q, is_suceed = image_quiz(g)

    if not is_suceed:
        continue


    divider = f'## 문제 {no}\n\n'
    print(divider)
    
    txt += divider
    # 파일명 추출해 이미지 링크 만들기
    filename = os.path.basename(g) # 마크다운에 표시할 이미지 파일 경로 설정   
    txt += f'![image](../img/{filename})\n\n'

    # 문제 추가
    print(q)
    txt += q + '\n\n---------------------\n\n'
    # 마크다운 파일로 저장
    with open('quiz/image_quiz_eng.md', 'w', encoding='utf-8') as f:
        f.write(txt)

    # 영어 문제만 추출
    eng = q.split('Listening: ')[1].split('정답:')[0].strip()

    eng_dict.append({
        'no': no,
        'eng': eng,
        'img': filename
    })

    # json 파일로 저장
    with open('quiz/image_quiz_eng.json', 'w', encoding='utf-8') as f:
        json.dump(eng_dict, f, ensure_ascii=False, indent=4)
    
    
    no += 1 # 문제 번호 증가

## 문제 1


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?  
- (1) 눈과 빙하로 덮인 높은 산들이 보입니다.  
- (2) 호수는 이미지의 왼쪽에 있습니다.  
- (3) 이미지 아래쪽 중앙 부근에 작은 산장이 있습니다.  
- (4) 전경에 나무와 초목이 펼쳐져 있습니다.  

Listening: Which of the following descriptions of the image is incorrect?  
- (1) There are high mountains covered with snow and glaciers.  
- (2) The lake is on the left side of the image.  
- (3) There is a small cabin near the bottom center of the image.  
- (4) Trees and vegetation spread across the foreground.  

정답: **(2)** 호수는 이미지의 왼쪽이 아니라 **오른쪽 아래쪽**에 있습니다.
## 문제 2


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 많은 사람들이 긴 테이블에 앉아 노트북을 사용하고 있습니다.
- (2) 대형 발표 화면은 행사장 오른쪽에 설치되어 있습니다.
- (3) 테이블 대부분은 검은색 천으로 덮여 있습니다.
- (4) 행사장 곳곳에 “DIVE 2024”라는 문구가 보입니다.

Listening: Which of the following descriptions of the image is incorrect?

- (1) Many people are seated at long tables using laptops.
- (2) A large presentation screen is set up on the right side of the venue.
- (3) Most of the tables are covered with black cloth.
- 

## TTS

In [8]:
response = client.audio.speech.create(
    model="tts-1-hd",
    voice="alloy",
    input="Hello world! This is a TTS test.",
)

response.write_to_file("audio/tts_test_alloy.mp3") # 파일에 쓰기

# 재생
import IPython.display as ipd

ipd.Audio("audio/tts_test_alloy.mp3")

In [16]:
# 다른 목소리
voice = "ash"
mp3_file = f"audio/tts_test_{voice}.mp3"

response = client.audio.speech.create(
    model="tts-1-hd",
    voice=voice,
    input=f"Hello world! I'm {voice}. This is a TTS test.",
)

response.write_to_file(mp3_file)

# 재생
import IPython.display as ipd

ipd.Audio(mp3_file)

## 한글 테스트

In [35]:
voice = "verse"
mp3_file = f"audio/tts_test_{voice}_ko.mp3"

response = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice=voice,

    # 말투/연기 지시
    instructions="""
    매우 밝고 에너지 넘치는 개그맨처럼 말하세요.
    감정을 크게 표현하고, 놀라거나 감탄하는 부분에서는 과장된 리액션을 해주세요.
    빠르고 경쾌한 템포로 말하세요.
    """,

    # 실제로 읽을 내용
    input="여러분 저 됐어요~"
)   

response.write_to_file(mp3_file)

# 재생
import IPython.display as ipd

ipd.Audio(mp3_file)

In [12]:
# json 파일 열기
with open('quiz/image_quiz_eng.json', 'r', encoding='utf-8') as f:
    eng_dict = json.load(f)

eng_dict

[{'no': 1,
  'eng': 'Which of the following descriptions of the image is incorrect?  \n- (1) There are high mountains covered with snow and glaciers.  \n- (2) The lake is on the left side of the image.  \n- (3) There is a small cabin near the bottom center of the image.  \n- (4) Trees and vegetation spread across the foreground.',
  'img': 'alex-ramon-aY_nzJTardo-unsplash.jpg'},
 {'no': 2,
  'eng': 'Which of the following descriptions of the image is incorrect?\n\n- (1) Many people are seated at long tables using laptops.\n- (2) A large presentation screen is set up on the right side of the venue.\n- (3) Most of the tables are covered with black cloth.\n- (4) The words “DIVE 2024” can be seen in several places around the venue.',
  'img': 'busan_dive.jpg'},
 {'no': 3,
  'eng': 'Which of the following descriptions of the image is incorrect?  \n- (1) Several pine trees can be seen in the foreground.  \n- (2) All the trees are bare, with no leaves on them.  \n- (3) A mountain range stretc

In [14]:
voices = ['alloy', 'ash', 'coral', 'echo', 'fable', 'onyx', 'nova', 'sage' , 'shimmer']

for q in eng_dict:
    no = q['no']
    quiz = q['eng']
    quiz = quiz.replace("- (1)", "- One.\t")
    quiz = quiz.replace("- (2)", "- Two.\t")
    quiz = quiz.replace("- (3)", "- Three.\t")
    quiz = quiz.replace("- (4)", "- Four.\t")    

    print(no, quiz)
    
    voice = voices[no % len(voices)] # 문제 개수를 목소리 개수로 나눈 나머지 값으로 선택  

    response = client.audio.speech.create(
        model="tts-1-hd",
        voice=voice,
        input=f'#{no}. {quiz}',
    )

    response.write_to_file(f"audio/{no}.mp3")

1 Which of the following descriptions of the image is incorrect?  
- One.	 There are high mountains covered with snow and glaciers.  
- Two.	 The lake is on the left side of the image.  
- Three.	 There is a small cabin near the bottom center of the image.  
- Four.	 Trees and vegetation spread across the foreground.
2 Which of the following descriptions of the image is incorrect?

- One.	 Many people are seated at long tables using laptops.
- Two.	 A large presentation screen is set up on the right side of the venue.
- Three.	 Most of the tables are covered with black cloth.
- Four.	 The words “DIVE 2024” can be seen in several places around the venue.
3 Which of the following descriptions of the image is incorrect?  
- One.	 Several pine trees can be seen in the foreground.  
- Two.	 All the trees are bare, with no leaves on them.  
- Three.	 A mountain range stretches out behind the trees.  
- Four.	 The sky is blue, with some white clouds visible.


In [15]:
ipd.Audio(f"audio/1.mp3")